# 🔬 t-SNE: t-Distributed Stochastic Neighbor Embedding
## A Complete, Beginner-Friendly Implementation

---

### 📦 Datasets Used
| Dataset | Description | Link |
|---------|-------------|------|
| **Iris** | 150 samples, 4 features, 3 species classes | `sklearn.datasets.load_iris()` |
| **MNIST (subset)** | Handwritten digits 0–9, 28×28 pixels → 784 features | `sklearn.datasets.fetch_openml('mnist_784')` |

### 🎯 What You'll Learn
1. How t-SNE transforms high-dimensional data into 2D
2. How to interpret t-SNE visualizations correctly
3. The effect of key hyperparameters (perplexity, learning rate)
4. How t-SNE compares to PCA visually

### ⏱️ Estimated Runtime: ~3–5 minutes (MNIST section is slowest)

---
> 💡 **Pro tip:** Run all cells in order. Each cell builds on the previous one.

---
## Cell 2 — 📦 Imports

We'll use the following libraries:

| Library | Purpose |
|---------|--------|
| `numpy` | Numerical operations and array manipulation |
| `pandas` | Data loading and exploration |
| `matplotlib` | Core plotting library |
| `seaborn` | Statistical visualizations (built on matplotlib) |
| `sklearn.manifold.TSNE` | The main algorithm we're learning |
| `sklearn.decomposition.PCA` | For comparison with t-SNE |
| `sklearn.preprocessing` | For normalizing features before t-SNE |
| `sklearn.datasets` | Pre-packaged datasets for experiments |

In [ ]:
# Core data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Scikit-learn: algorithms
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, fetch_openml
from sklearn.metrics import silhouette_score

# Utilities
import warnings
import time
warnings.filterwarnings('ignore')

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
RANDOM_STATE = 42  # Always set for reproducibility!

print("✅ All libraries imported successfully!")
print(f"   NumPy version : {np.__version__}")
print(f"   Pandas version: {pd.__version__}")
import sklearn; print(f"   Sklearn version: {sklearn.__version__}")

---
## Cell 3 — 🧠 Theory Recap: t-SNE in Plain English

### The Problem t-SNE Solves
Imagine you have a dataset with 784 features (like MNIST images). You can't visualize that.
But you want to understand: *"Do similar images live close together in feature space?"*

t-SNE answers this by:
1. Measuring **who is close to whom** in the original high-dimensional space using **Gaussian probabilities**
2. Creating a **2D map** where it tries to reproduce those same neighborhoods
3. Using a **Student's t-distribution** in 2D (instead of Gaussian) to avoid the "crowding problem"
4. Optimizing by minimizing **KL divergence** between the two probability distributions

### The 3 Core Equations (simplified)

**High-dim similarity:**
$$p_{ij} = \text{How likely are } x_i \text{ and } x_j \text{ to be neighbors in original space?}$$

**Low-dim similarity:**
$$q_{ij} = \text{How likely are } y_i \text{ and } y_j \text{ to be neighbors in 2D map?}$$

**Optimization goal:**
$$\text{Minimize } KL(P \| Q) = \sum_{i,j} p_{ij} \log\frac{p_{ij}}{q_{ij}}$$

### ⚠️ Critical Interpretability Rules

| Rule | Explanation |
|------|-------------|
| ✅ Within-cluster distances | Meaningful — nearby points are genuinely similar |
| ❌ Between-cluster distances | **Not meaningful** — cluster positions are arbitrary |
| ❌ Cluster sizes | Not proportional to real data density |
| ❌ Axes (x, y) | Have no interpretable meaning |
| ✅ Cluster existence | If a cluster appears consistently across runs, it's likely real |


---
## Cell 4 — 📊 Load Dataset: Iris

We start with the **Iris dataset** — a classic, small, well-understood dataset perfect for learning:
- 150 samples, 4 features, 3 classes (Setosa, Versicolor, Virginica)
- We *know* the true structure, so we can verify if t-SNE finds it

In [ ]:
# ── Load Iris Dataset ──────────────────────────────────────────────────────────
iris = load_iris()
X_iris = iris.data           # Shape: (150, 4)
y_iris = iris.target         # Labels: 0, 1, 2
target_names = iris.target_names  # ['setosa', 'versicolor', 'virginica']
feature_names = iris.feature_names

# ── Create a DataFrame for exploration ────────────────────────────────────────
df_iris = pd.DataFrame(X_iris, columns=feature_names)
df_iris['species'] = [target_names[i] for i in y_iris]

print("=" * 55)
print("         IRIS DATASET OVERVIEW")
print("=" * 55)
print(f"  Shape           : {X_iris.shape}  (samples × features)")
print(f"  Features        : {feature_names}")
print(f"  Classes         : {list(target_names)}")
print(f"  Samples/class   : {np.bincount(y_iris).tolist()}")
print("=" * 55)
print()
print("First 5 rows:")
display(df_iris.head())

print("\nBasic statistics:")
display(df_iris.describe().round(2))

---
## Cell 5 — 🔧 Preprocessing

**Why preprocess before t-SNE?**

t-SNE computes **Euclidean distances** between all pairs of points. If one feature has values in the range [0, 1000] and another in [0, 1], the first feature will dominate the distance computation — making the results misleading.

**Solution:** Standardize all features to have **mean = 0** and **std = 1** using `StandardScaler`.

**Optional but recommended:** If data has 50+ dimensions, apply PCA first to reduce to ~50 dimensions. This:
- Removes noise from low-variance components
- Speeds up t-SNE significantly (O(N²) scaling makes feature count impactful)
- Is exactly what `sklearn`'s t-SNE does internally when `init='pca'`

In [ ]:
# ── Step 1: Standardize features ──────────────────────────────────────────────
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

print("Before scaling:")
print(f"  Mean  : {X_iris.mean(axis=0).round(2)}")
print(f"  Std   : {X_iris.std(axis=0).round(2)}")
print()
print("After scaling:")
print(f"  Mean  : {X_iris_scaled.mean(axis=0).round(6)}  (≈ 0)")
print(f"  Std   : {X_iris_scaled.std(axis=0).round(6)}   (≈ 1)")

# ── Visualize feature distributions before/after ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, name in enumerate(feature_names):
    axes[0].hist(X_iris[:, i], bins=20, alpha=0.6, label=name)
    axes[1].hist(X_iris_scaled[:, i], bins=20, alpha=0.6, label=name)

axes[0].set_title('Feature Distributions — Before Scaling', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature Value')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].set_title('Feature Distributions — After Scaling (StandardScaler)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature Value (Standardized)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Preprocessing complete. Data ready for t-SNE.")

---
## Cell 6 — 🔍 Understanding t-SNE Logic (Conceptual Deep Dive)

Before applying t-SNE, let's build intuition for what it's doing under the hood — without getting lost in math.

### The Neighborhood Analogy

Think of each data point as a **person** at a party. The 4D Iris feature space is the actual room layout. t-SNE asks:

> *"If I took a photo of this party from above and compressed it to a 2D seating chart, how do I preserve who's talking to whom?"*

### Step-by-Step Logic

```
1️⃣  MEASURE CLOSENESS (high-dim)
    For each pair (i, j), compute how "similar" they are.
    → Convert Euclidean distance to a probability using Gaussian
    → p_ij = "probability that i would choose j as a neighbor"
    → Perplexity controls how many neighbors each point considers

2️⃣  RANDOMLY PLACE IN 2D
    Start with random (or PCA-initialized) 2D coordinates
    → These are just the starting positions for optimization

3️⃣  MEASURE 2D CLOSENESS
    Compute similarities in 2D using t-distribution (NOT Gaussian)
    → q_ij = "probability that i would choose j as a neighbor in 2D"
    → Why t-distribution? Heavy tails push dissimilar points far apart

4️⃣  COMPARE AND ADJUST
    If p_ij > q_ij  →  pull i and j closer in 2D
    If p_ij < q_ij  →  push i and j apart in 2D
    Repeat 1000 times (gradient descent)
```

### Why the t-distribution fixes the crowding problem

Imagine fitting 150 people from a 4D room into a 2D floor plan.
- In 4D, points can be "moderately far" in many directions
- In 2D, there's just not enough room to represent all moderate distances faithfully
- **Gaussian** would squish everyone together in the center (crowding!)
- **t-distribution** has heavier tails → allows moderately similar points to spread out more naturally

**Think of it as:** Gaussian says "close is close, medium is medium-close." t-distribution says "close is close, medium is FAR." This dramatic stretching creates the visually clean clusters we love in t-SNE plots.

In [ ]:
# ── Visualize the key difference: Gaussian vs t-distribution ──────────────────
# This is the core reason t-SNE works better than SNE

x = np.linspace(-5, 5, 500)
gaussian = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
t_dist   = (1 + x**2)**(-1) / np.pi  # Cauchy = t-dist with 1 degree of freedom

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Distribution shapes ────────────────────────────────────────────────
ax1.plot(x, gaussian, 'b-', linewidth=2.5, label='Gaussian (used in high-dim)')
ax1.plot(x, t_dist,   'r-', linewidth=2.5, label='t-distribution (used in 2D)')
ax1.fill_between(x, 0, gaussian, alpha=0.15, color='blue')
ax1.fill_between(x, 0, t_dist,   alpha=0.15, color='red')
ax1.axvline(x=2, color='gray', linestyle='--', alpha=0.7, label='"Moderate distance"')
ax1.set_title('Gaussian vs t-Distribution:\nWhy t-SNE Uses t-Distribution in 2D', 
              fontsize=12, fontweight='bold')
ax1.set_xlabel('Distance between points')
ax1.set_ylabel('Similarity (probability)')
ax1.legend(fontsize=10)
ax1.annotate('Heavy tail →\ndissimilar points\nget pushed far apart',
             xy=(3.5, 0.03), fontsize=9, color='red',
             arrowprops=dict(arrowstyle='->', color='red'),
             xytext=(2.5, 0.15))

# ── Right: Crowding problem illustration ─────────────────────────────────────
np.random.seed(RANDOM_STATE)
n = 80
theta = np.linspace(0, 2*np.pi, n)
# Three concentric ring clusters
r1 = 1.0 + np.random.randn(n) * 0.08
r2 = 2.0 + np.random.randn(n) * 0.08
r3 = 3.0 + np.random.randn(n) * 0.08

for r, color, label in [(r1, 'blue', 'Cluster A'), 
                         (r2, 'orange', 'Cluster B'), 
                         (r3, 'green', 'Cluster C')]:
    ax2.scatter(r * np.cos(theta), r * np.sin(theta), 
                c=color, s=30, alpha=0.7, label=label)

ax2.set_title('The Crowding Problem Illustration:\nWith Gaussian, all clusters collapse toward center',
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.legend()
ax2.set_aspect('equal')
ax2.text(0, 0, 'Gaussian\npulls all\nhere', ha='center', va='center', 
         color='red', fontsize=9, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('gaussian_vs_tdist.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 The t-distribution's heavy tails let t-SNE push dissimilar points apart,")
print("   creating cleaner, more separated clusters in the 2D visualization.")

---
## Cell 7 — ⚙️ Apply t-SNE on Iris Dataset

Let's now run t-SNE and see what it produces.

In [ ]:
# ── Configure and run t-SNE ───────────────────────────────────────────────────
print("Running t-SNE on Iris dataset...")
print(f"Input shape: {X_iris_scaled.shape}")

tsne = TSNE(
    n_components=2,        # Reduce to 2D for visualization
    perplexity=30,         # Effective neighbors (try 5–50)
    learning_rate='auto',  # sklearn auto-selects optimal lr
    n_iter=1000,           # Optimization steps
    random_state=RANDOM_STATE,  # Reproducibility!
    init='pca',            # PCA init is more stable than 'random'
    verbose=1              # Show progress
)

start = time.time()
X_tsne = tsne.fit_transform(X_iris_scaled)
elapsed = time.time() - start

print(f"\n✅ t-SNE complete!")
print(f"   Input shape  : {X_iris_scaled.shape}")
print(f"   Output shape : {X_tsne.shape}")
print(f"   Time elapsed : {elapsed:.2f} seconds")
print(f"   Final KL divergence: {tsne.kl_divergence_:.4f}")
print()
print("First 5 embedded points (2D coordinates):")
for i in range(5):
    print(f"  Point {i}: ({X_tsne[i,0]:.3f}, {X_tsne[i,1]:.3f}) — {target_names[y_iris[i]]}")

---
## Cell 8 — 📈 Visualization: t-SNE 2D Plot

A well-labeled, publication-quality scatter plot.

**What to look for:**
- Do the three species form distinct clusters?
- Is Setosa (the most distinct species) clearly separated?
- Are Versicolor and Virginica mixed or separated? (They overlap in original feature space)

In [ ]:
# ── t-SNE Visualization ───────────────────────────────────────────────────────
colors   = ['#E74C3C', '#3498DB', '#2ECC71']  # Red, Blue, Green
markers  = ['o', 's', '^']                     # Circle, Square, Triangle

fig, ax = plt.subplots(figsize=(10, 7))

for idx, (species, color, marker) in enumerate(zip(target_names, colors, markers)):
    mask = y_iris == idx
    ax.scatter(
        X_tsne[mask, 0],
        X_tsne[mask, 1],
        c=color,
        marker=marker,
        s=100,
        alpha=0.85,
        edgecolors='white',
        linewidths=0.5,
        label=f'Iris {species.capitalize()}  (n={mask.sum()})'
    )

# ── Annotations ──────────────────────────────────────────────────────────────
ax.set_title(
    't-SNE Visualization of Iris Dataset (4D → 2D)\n'
    'Perplexity=30 | Learning Rate=auto | Iterations=1000',
    fontsize=14, fontweight='bold', pad=15
)
ax.set_xlabel('t-SNE Component 1 (no unit — axes are arbitrary)', fontsize=11)
ax.set_ylabel('t-SNE Component 2 (no unit — axes are arbitrary)', fontsize=11)
ax.legend(fontsize=11, framealpha=0.9, loc='upper right')

# Add interpretability warning
ax.text(
    0.02, 0.02,
    '⚠️ Note: Axes & inter-cluster distances are NOT meaningful.\n'
    '   Only within-cluster proximity matters.',
    transform=ax.transAxes,
    fontsize=9, color='gray',
    verticalalignment='bottom',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
)

plt.tight_layout()
plt.savefig('tsne_iris.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Observations:")
print("  • Setosa (red circles) forms a tight, well-separated cluster — very distinct species")
print("  • Versicolor (blue squares) and Virginica (green triangles) partially overlap")
print("  • This matches domain knowledge: Setosa is easily distinguishable, the others are not")
print("  • t-SNE successfully revealed the true structure of this 4D dataset in 2D!")

---
## Cell 9 — 🔄 Compare: t-SNE vs PCA

The most important comparison in dimensionality reduction.

**Key differences to observe:**
- **PCA** maximizes variance — it will show the direction of most spread
- **t-SNE** maximizes cluster separation — it will show the neighborhood structure
- For Iris, both work reasonably well, but t-SNE typically gives cleaner cluster separation

In [ ]:
# ── Run PCA for comparison ────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_iris_scaled)

explained_var = pca.explained_variance_ratio_
print(f"PCA explained variance: PC1={explained_var[0]*100:.1f}%, PC2={explained_var[1]*100:.1f}%")
print(f"Total explained: {sum(explained_var)*100:.1f}% of original variance captured")

# ── Side-by-side comparison ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for ax, X_embed, title, xlabel, ylabel in [
    (ax1, X_pca,
     f'PCA (Linear) — Iris Dataset\nPC1={explained_var[0]*100:.1f}% + PC2={explained_var[1]*100:.1f}% = {sum(explained_var)*100:.1f}% variance',
     f'PC1 ({explained_var[0]*100:.1f}% variance) — Interpretable!',
     f'PC2 ({explained_var[1]*100:.1f}% variance) — Interpretable!'),
    (ax2, X_tsne,
     't-SNE (Non-linear) — Iris Dataset\nPerplexity=30, Iterations=1000',
     't-SNE Component 1 (arbitrary, no unit)',
     't-SNE Component 2 (arbitrary, no unit)')
]:
    for idx, (species, color, marker) in enumerate(zip(target_names, colors, markers)):
        mask = y_iris == idx
        ax.scatter(X_embed[mask, 0], X_embed[mask, 1],
                   c=color, marker=marker, s=80, alpha=0.8,
                   edgecolors='white', linewidths=0.5,
                   label=f'Iris {species.capitalize()}')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=10)

plt.suptitle('PCA vs t-SNE: Same Data, Different Perspectives', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pca_vs_tsne_iris.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Quantitative comparison: Silhouette Score ─────────────────────────────────
s_pca   = silhouette_score(X_pca,   y_iris)
s_tsne  = silhouette_score(X_tsne,  y_iris)

print("\n── Quantitative Cluster Quality (Silhouette Score: higher = better) ──")
print(f"  PCA  silhouette score: {s_pca:.4f}")
print(f"  t-SNE silhouette score: {s_tsne:.4f}")
winner = 't-SNE' if s_tsne > s_pca else 'PCA'
print(f"  → {winner} produces better-separated clusters for this dataset")

print("\n── Key Differences ──")
comparison = {
    'Technique': ['PCA', 't-SNE'],
    'Type': ['Linear', 'Non-linear'],
    'Axes meaningful?': ['✅ Yes (variance)', '❌ No (arbitrary)'],
    'New data?': ['✅ Yes', '❌ No (must refit)'],
    'Best for': ['Feature reduction, preprocessing', 'Visualization, EDA']
}
display(pd.DataFrame(comparison))

---
## Cell 10 — 🎛️ Hyperparameter Experiment: Perplexity

**Perplexity** is the single most important hyperparameter in t-SNE. It controls:
- How many effective neighbors each point considers
- The width of the Gaussian kernel (σᵢ)
- The balance between local and global structure

### What to expect:
| Perplexity | Effect |
|-----------|--------|
| Very low (2–5) | Tight micro-clusters, noisy, may split real clusters |
| Good range (15–50) | Meaningful neighborhood structure |
| Very high (>100) | Global structure only, clusters merge into blobs |
| ≥ N | Breaks down mathematically |

**Rule of thumb:** Start with `sqrt(N)` and explore ±50% of that value.

> ⏱️ This cell runs t-SNE 6 times — takes ~30 seconds.

In [ ]:
# ── Perplexity Experiment ─────────────────────────────────────────────────────
perplexity_values = [2, 5, 15, 30, 50, 100]
tsne_results = {}
kl_divergences = {}

print("Running t-SNE with different perplexity values...")
print(f"Dataset: Iris (N={len(X_iris_scaled)}, recommended perplexity: 5–50)")
print()

for perp in perplexity_values:
    # Skip invalid perplexity (must be < n_samples)
    if perp >= len(X_iris_scaled):
        print(f"  Perplexity={perp:>4}: ⛔ Skipped (must be < n_samples={len(X_iris_scaled)})")
        tsne_results[perp] = None
        continue
    
    t0 = time.time()
    tsne_exp = TSNE(
        n_components=2,
        perplexity=perp,
        learning_rate='auto',
        n_iter=1000,
        random_state=RANDOM_STATE,
        init='pca'
    )
    result = tsne_exp.fit_transform(X_iris_scaled)
    elapsed = time.time() - t0
    
    tsne_results[perp] = result
    kl_divergences[perp] = tsne_exp.kl_divergence_
    
    print(f"  Perplexity={perp:>4}: ✅ Done | KL divergence={tsne_exp.kl_divergence_:.4f} | Time={elapsed:.1f}s")

print("\n✅ All perplexity experiments complete!")

---
## Cell 11 — 📊 Plot Perplexity Comparison

Visualizing how cluster structure changes with perplexity.

In [ ]:
# ── Plot all perplexity results ───────────────────────────────────────────────
valid_perps = [p for p in perplexity_values if tsne_results[p] is not None]
n_plots = len(valid_perps)
cols = 3
rows = (n_plots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
axes = axes.flatten()

quality_labels = {
    2:   '⚠️ Too Low — Micro-clusters, noisy',
    5:   '⚠️ Low — Some structure visible',
    15:  '✅ Good — Clear local structure',
    30:  '✅ Optimal — Well-separated clusters',
    50:  '✅ Good — Slightly more global',
    100: '⚠️ High — Clusters start merging'
}

for i, perp in enumerate(valid_perps):
    ax = axes[i]
    X_embed = tsne_results[perp]
    
    for idx, (species, color, marker) in enumerate(zip(target_names, colors, markers)):
        mask = y_iris == idx
        ax.scatter(X_embed[mask, 0], X_embed[mask, 1],
                   c=color, marker=marker, s=60, alpha=0.85,
                   edgecolors='white', linewidths=0.4,
                   label=species.capitalize())
    
    ax.set_title(
        f'Perplexity = {perp}\n'
        f'{quality_labels.get(perp, "")}\n'
        f'KL Divergence: {kl_divergences[perp]:.4f}',
        fontsize=10, fontweight='bold'
    )
    ax.set_xlabel('t-SNE 1', fontsize=9)
    ax.set_ylabel('t-SNE 2', fontsize=9)
    ax.legend(fontsize=8, loc='upper right')
    ax.tick_params(labelsize=8)

# Hide any unused subplots
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Effect of Perplexity on t-SNE: Iris Dataset\n'
    'Lower perplexity = local focus | Higher perplexity = global focus',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('tsne_perplexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── KL Divergence Plot ────────────────────────────────────────────────────────
fig2, ax = plt.subplots(figsize=(9, 4))
perp_vals = list(kl_divergences.keys())
kl_vals   = list(kl_divergences.values())

ax.plot(perp_vals, kl_vals, 'bo-', linewidth=2, markersize=9)
for px, kl in zip(perp_vals, kl_vals):
    ax.annotate(f'{kl:.3f}', (px, kl), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)

ax.set_title('KL Divergence vs Perplexity\n(Lower KL = better match between high-dim and 2D probabilities)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Perplexity', fontsize=11)
ax.set_ylabel('KL Divergence (lower = better)', fontsize=11)
ax.set_xticks(perp_vals)
ax.fill_between([15, 50], ax.get_ylim()[0], ax.get_ylim()[1],
                alpha=0.1, color='green', label='Recommended range (15–50)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('kl_divergence_vs_perplexity.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Key observation: KL divergence alone doesn't tell the full story.")
print("   A lower KL is better, but also inspect the plots visually.")
print("   Structure that appears in ALL runs is likely real. Structure that")
print("   appears only at one perplexity may be an artifact.")

---
## Bonus: t-SNE on MNIST (High-Dimensional Data)

Iris has only 4 features — t-SNE's power really shines on high-dimensional data.
Let's run it on **MNIST** (784 features = 28×28 pixel images of handwritten digits).

We'll use a subset of 3,000 samples to keep runtime manageable.

> ⏱️ This takes ~60–90 seconds. Worth it!
>
> Note: First run downloads MNIST (~12 MB). Subsequent runs use cache.

In [ ]:
# ── Load MNIST ────────────────────────────────────────────────────────────────
print("Loading MNIST dataset (28×28 handwritten digit images)...")
print("This may take a moment on first run (downloading ~12 MB)...")

mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_mnist = mnist.data.astype(np.float32)   # Shape: (70000, 784)
y_mnist = mnist.target.astype(int)         # Labels: 0–9

print(f"\n✅ MNIST loaded!")
print(f"   Full dataset shape: {X_mnist.shape}")
print(f"   Labels: digits 0–9, distribution: {np.bincount(y_mnist).tolist()}")

# ── Subsample for speed ───────────────────────────────────────────────────────
N_SUBSET = 3000
np.random.seed(RANDOM_STATE)
idx = np.random.choice(len(X_mnist), N_SUBSET, replace=False)
X_sub = X_mnist[idx]
y_sub = y_mnist[idx]

# Normalize to [0, 1]
X_sub = X_sub / 255.0

print(f"\n   Using subset: {N_SUBSET} samples (for speed)")
print(f"   Subset class distribution: {np.bincount(y_sub).tolist()}")

# ── Visualize some MNIST samples ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
plt.suptitle('MNIST Sample Images (What t-SNE will embed into 2D)', 
             fontsize=13, fontweight='bold')
for digit in range(10):
    # Find examples of each digit
    examples = np.where(y_sub == digit)[0][:2]
    for row, ex_idx in enumerate(examples):
        axes[row, digit].imshow(X_sub[ex_idx].reshape(28, 28), cmap='gray')
        axes[row, digit].set_title(f'Label: {digit}', fontsize=8)
        axes[row, digit].axis('off')

plt.tight_layout()
plt.savefig('mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── PCA pre-reduction (standard practice for high-dim data) ──────────────────
print("Step 1: Reducing MNIST from 784 → 50 dims with PCA (standard preprocessing)")
pca_mnist = PCA(n_components=50, random_state=RANDOM_STATE)
X_sub_pca50 = pca_mnist.fit_transform(X_sub)
var_retained = sum(pca_mnist.explained_variance_ratio_) * 100
print(f"  Variance retained: {var_retained:.1f}% (50 PCA components)")

print("\nStep 2: Applying t-SNE on 50-dim PCA output...")
print(f"  Input: {X_sub_pca50.shape} | Output: (3000, 2)")

tsne_mnist = TSNE(
    n_components=2,
    perplexity=40,
    learning_rate='auto',
    n_iter=1000,
    random_state=RANDOM_STATE,
    init='pca',
    verbose=1
)

t0 = time.time()
X_mnist_tsne = tsne_mnist.fit_transform(X_sub_pca50)
print(f"\n✅ Done! Time: {time.time()-t0:.1f}s | KL divergence: {tsne_mnist.kl_divergence_:.4f}")

In [ ]:
# ── MNIST t-SNE Visualization ─────────────────────────────────────────────────
digit_colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# ── Left: Colored by digit class ─────────────────────────────────────────────
scatter = ax1.scatter(
    X_mnist_tsne[:, 0],
    X_mnist_tsne[:, 1],
    c=y_sub,
    cmap='tab10',
    s=12,
    alpha=0.7,
    linewidths=0
)
cbar = plt.colorbar(scatter, ax=ax1, ticks=range(10))
cbar.set_label('Digit Class', fontsize=11)
cbar.set_ticklabels([str(i) for i in range(10)])

ax1.set_title(
    f't-SNE on MNIST (784D → 2D)\n'
    f'{N_SUBSET} samples, Perplexity=40, Iterations=1000',
    fontsize=13, fontweight='bold'
)
ax1.set_xlabel('t-SNE Component 1 (arbitrary)', fontsize=11)
ax1.set_ylabel('t-SNE Component 2 (arbitrary)', fontsize=11)

# Add digit labels at cluster centroids
for digit in range(10):
    mask = y_sub == digit
    cx = X_mnist_tsne[mask, 0].mean()
    cy = X_mnist_tsne[mask, 1].mean()
    ax1.text(cx, cy, str(digit), fontsize=14, fontweight='bold',
             ha='center', va='center', color='black',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

# ── Right: PCA 2D for comparison ──────────────────────────────────────────────
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_mnist_pca2 = pca_2d.fit_transform(X_sub)

scatter2 = ax2.scatter(
    X_mnist_pca2[:, 0],
    X_mnist_pca2[:, 1],
    c=y_sub,
    cmap='tab10',
    s=12,
    alpha=0.7,
    linewidths=0
)
plt.colorbar(scatter2, ax=ax2, ticks=range(10)).set_label('Digit Class', fontsize=11)

ev = pca_2d.explained_variance_ratio_
ax2.set_title(
    f'PCA on MNIST (784D → 2D)\n'
    f'PC1={ev[0]*100:.1f}% + PC2={ev[1]*100:.1f}% = {sum(ev)*100:.1f}% variance',
    fontsize=13, fontweight='bold'
)
ax2.set_xlabel(f'PC1 ({ev[0]*100:.1f}% variance)', fontsize=11)
ax2.set_ylabel(f'PC2 ({ev[1]*100:.1f}% variance)', fontsize=11)

plt.suptitle('t-SNE vs PCA on MNIST Handwritten Digits\n'
             't-SNE reveals 10 clearly separated digit clusters; PCA overlaps heavily',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('tsne_vs_pca_mnist.png', dpi=150, bbox_inches='tight')
plt.show()

# Silhouette
s_mnist_tsne = silhouette_score(X_mnist_tsne, y_sub)
s_mnist_pca  = silhouette_score(X_mnist_pca2, y_sub)
print(f"\n── Silhouette Scores (MNIST subset) ──")
print(f"  t-SNE: {s_mnist_tsne:.4f}")
print(f"  PCA:   {s_mnist_pca:.4f}")
print(f"  → t-SNE outperforms PCA by {(s_mnist_tsne/s_mnist_pca - 1)*100:.0f}% on cluster separation")

---
## Cell 12 — 🏁 Key Takeaways

### What We Learned

In [ ]:
# ── Final Summary Dashboard ───────────────────────────────────────────────────
print("=" * 65)
print("         KEY TAKEAWAYS: t-SNE Complete Guide")
print("=" * 65)

takeaways = [
    ("1", "WHAT",
     "t-SNE is a non-linear dimensionality reduction algorithm\n"
     "     for VISUALIZATION. It maps high-dim data to 2D/3D while\n"
     "     preserving local neighborhood structure."),
    
    ("2", "HOW",
     "Converts distances → probabilities (Gaussian in high-dim,\n"
     "     t-distribution in low-dim), then minimizes KL divergence\n"
     "     between the two distributions via gradient descent."),
    
    ("3", "WHY t-DIST",
     "The Student's t-distribution has heavy tails, which resolves\n"
     "     the crowding problem — letting dissimilar points spread\n"
     "     apart in the 2D map for cleaner visualization."),
    
    ("4", "PERPLEXITY",
     "The most important hyperparameter. Controls effective number\n"
     "     of neighbors. Use 5–50 for most datasets. Always try\n"
     "     multiple values and verify structure is consistent."),
    
    ("5", "INTERPRET",
     "✅ Within-cluster proximity IS meaningful\n"
     "     ❌ Between-cluster distances are NOT meaningful\n"
     "     ❌ Cluster sizes do NOT reflect real data density\n"
     "     ❌ Axes (x, y) have NO unit or meaning"),
    
    ("6", "LIMITS",
     "No transform() — cannot embed new data without refitting.\n"
     "     Non-deterministic — set random_state for reproducibility.\n"
     "     Slow on >100K samples — use UMAP instead."),
    
    ("7", "vs PCA",
     "PCA: linear, interpretable axes, invertible, fast.\n"
     "     t-SNE: non-linear, better clusters, not invertible, slow.\n"
     "     Use PCA for preprocessing; t-SNE for visualization."),
    
    ("8", "WORKFLOW",
     "Scale → (PCA to 50 dims if needed) → t-SNE → Visualize.\n"
     "     Validate clusters quantitatively (silhouette score).\n"
     "     Run multiple seeds/perplexities before concluding."),
]

for num, topic, detail in takeaways:
    print(f"\n  [{num}] {topic}")
    print(f"     {detail}")

print()
print("=" * 65)
print("  📁 Files generated in this notebook:")
print("     • feature_distributions.png    — Before/after scaling")
print("     • gaussian_vs_tdist.png        — Core t-SNE insight")
print("     • tsne_iris.png                — Main t-SNE result")
print("     • pca_vs_tsne_iris.png         — Algorithm comparison")
print("     • tsne_perplexity_comparison.png — Hyperparameter study")
print("     • kl_divergence_vs_perplexity.png — Quantitative metric")
print("     • mnist_samples.png            — MNIST sample images")
print("     • tsne_vs_pca_mnist.png        — High-dim comparison")
print("=" * 65)
print()
print("  🎯 Next Steps:")
print("     1. Try t-SNE on your own dataset")
print("     2. Experiment with learning_rate (try 10, 100, 500, 1000)")
print("     3. Compare with UMAP (faster, supports transform())")
print("     4. Apply to neural network embeddings (word2vec, BERT, ResNet)")
print("     5. Read: https://distill.pub/2016/misread-tsne/")
print()
print("  📚 Reference: van der Maaten & Hinton (2008), JMLR")
print("=" * 65)

---

## 📋 Quick Cheat Sheet

```python
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# 1. Always scale first
X_scaled = StandardScaler().fit_transform(X)

# 2. PCA pre-reduction for high-dim data (>50 features)
# X_scaled = PCA(n_components=50).fit_transform(X_scaled)

# 3. Apply t-SNE
tsne = TSNE(
    n_components=2,        # 2 for visualization
    perplexity=30,         # try: 5, 15, 30, 50
    learning_rate='auto',  # or try: 10–1000
    n_iter=1000,           # more = better, slower
    random_state=42,       # ALWAYS set this!
    init='pca'             # more stable than 'random'
)
X_2d = tsne.fit_transform(X_scaled)

# 4. Visualize
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10')
plt.title('t-SNE Visualization')
plt.colorbar()
plt.show()

# ⚠️ Remember: NO transform() for new data — must refit!
# ⚠️ Remember: Inter-cluster distances are NOT meaningful!
```

---
*Notebook built for Gradientts ML Internship Program — t-SNE Module*  
*Dataset sources: UCI ML Repository (Iris) | Yann LeCun's MNIST Database*